# JCSD: GPU smoke test + hardware profile
Works on Colab and Kaggle (GPU runtime, fp16). Upload `jcsd.zip` (made with `zip -r jcsd.zip jcsd -x '*/__pycache__/*' '*.pytest_cache*'`) from the project folder.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.get_device_name())

In [ ]:
# Colab: upload the zip. Kaggle: add it as a dataset and set ZIP to its path instead.
ZIP = 'jcsd.zip'
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass
!unzip -oq {ZIP}
%cd jcsd

In [ ]:
!pip -q install transformers accelerate datasets evaluate rouge_score scipy pytest
import transformers; print(transformers.__version__)

## 1. Unit tests on the GPU machine
The slow chi-square tests take ~2 minutes on CPU; skip with `-k 'not output_matches'` if short on time.

In [ ]:
!python -m pytest -q

## 2. Tree-attention support check (cheap, do before profiling)
Tiny check that the model family accepts an explicit 4D mask + position ids. The full numerical check (spec 8.3) comes with `tree.py`.

In [ ]:
import torch
from jcsd.config import Cfg
from jcsd.models import load_pair
from jcsd import profile_hw as ph
cfg = Cfg()   # pythia-410m / pythia-2.8b, fp16, sdpa
tok, draft, target, V = load_pair(cfg)
print('V =', V)
ph.time_forward(target, 64, 8, 'cuda', masked=True, reps=3, warm=1)
print('masked call ran')

## 3. Profile this platform
`--tag` labels the platform. Use a different tag for a different precision/offload setup and say so in the write-up. Run plain first, then masked.

In [ ]:
!python scripts/run_profile.py --ctx 128 512 --tag $(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')_fp16
!python scripts/run_profile.py --ctx 128 512 --masked --tag $(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')_fp16

In [ ]:
!zip -qr profiles.zip profiles
try:
    from google.colab import files; files.download('profiles.zip')
except ImportError:
    print('download profiles.zip from the Kaggle output panel')